## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [2]:
import statsmodels.stats.api as sms

# очікувані пропорції (19% vs 18%)
effect_size = sms.proportion_effectsize(0.19, 0.18)

required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)

print(f"Необхідний розмір вибірки на групу: {int(required_n)}")

Необхідний розмір вибірки на групу: 23664


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [3]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Analytics_HW/cookie_cats.csv')

df.groupby('version')['retention_7'].mean()

,retention_7
version,
gate_30,0.190201
gate_40,0.182000


Гіпотези:

H₀: p₃₀ = p₄₀ (утримання однакове)

Hₐ: p₃₀ ≠ p₄₀ (одна версія дає краще retention)

З середніх видно:

gate_30 ≈ 0.190
gate_40 ≈ 0.182

Попередньо виглядає, що gate_30 працює краще.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [4]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

results = df.groupby('version')['retention_7'].agg(['sum','count'])

successes = results['sum'].values
nobs = results['count'].values

z_stat, p_value = proportions_ztest(successes, nobs)

(lower_30, lower_40), (upper_30, upper_40) = proportion_confint(
    successes,
    nobs,
    alpha=0.05
)

print("🔹 A/B test retention_7")

print(f"Z-статистика = {z_stat:.3f}")
print(f"p-value = {p_value:.5f}")

print("\n95% довірчі інтервали:")

print(f"gate_30: ({lower_30:.3f}, {upper_30:.3f})")
print(f"gate_40: ({lower_40:.3f}, {upper_40:.3f})")

🔹 A/B test retention_7
Z-статистика = 3.164
p-value = 0.00155

95% довірчі інтервали:
gate_30: (0.187, 0.194)
gate_40: (0.178, 0.186)


Висновок:

Оскільки:

p-value ≈ 0.00155 < 0.05

ми відхиляємо H₀.

1) Є статистично значущі підстави вважати, що версії гри дають різний retention.

Оскільки:

retention gate_30 > gate_40

можна зробити висновок, що gate_30 показує кращий результат утримання гравців.

2) Довірчі інтервали не перетинаються. Це означає, що різниця між retention у версіях gate_30 та gate_40 є статистично значущою і версія gate_30 має кращий показник утримання користувачів.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [5]:
import scipy.stats as stats

crosstab = pd.crosstab(df['version'], df['retention_7'])

chi2, p, dof, expected = stats.chi2_contingency(crosstab)

print("🔹 Chi-square test")

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")

print("\nОчікувані частоти:")
print(expected)

🔹 Chi-square test
χ² = 9.959
p-value = 0.00160
Ступені свободи = 1

Очікувані частоти:
[[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]


Гіпотези для Chi-square:

H₀: version і retention незалежні
Hₐ: version і retention залежні

Висновок:
Оскільки:

p-value ≈ 0.0016 < 0.05

ми відхиляємо H₀.

Це означає, що існує статистично значуща залежність між версією гри та retention.

Фінальний бізнес-висновок:

Статистичні тести (Z-test та Chi-square) показали, що версія gate_30 забезпечує краще утримання користувачів, ніж gate_40. Різниця є статистично значущою.